# 02 — Preprocessing & Feature Extraction (all 3 levels)

Loads the WMMD HuggingFace dataset (cached at `Mss/data/wmms_parquet_local`),
subsets per level, runs the preprocessing pipeline (resample → 2-sec windows →
bandpass), extracts features and log-Mel spectrograms, and writes the `.npz`
files specified in `PROJECT_CONTEXT.txt`.

**Levels:**
- **Level 1** — 4 family-level groups (Phocidae, Monodontidae, Balaenidae, Delphinidae subset). 32-dim features.
- **Level 2** — All Delphinidae species. 51-dim features.
- **Level 3** — Spinner Dolphin only. 46-dim features, no labels.


In [ ]:
import io
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from datasets import load_from_disk, Audio

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.preprocessing import preprocess_clip, TARGET_SR
from utils.features import (
    extract_features_level1,
    extract_features_level2,
    extract_features_level3,
    compute_mel_spectrogram,
)

print("project root:", PROJECT_ROOT)


## Subsetting rules per level

These mirror the tables in `PROJECT_CONTEXT.txt`. Note that a single clip can
appear in multiple levels (e.g. all Spinner Dolphin clips appear in both
Level 2 and Level 3).


In [ ]:
HF_LOCAL = PROJECT_ROOT / "Mss" / "data" / "wmms_parquet_local"
METADATA_CSV = PROJECT_ROOT / "wmmd_metadata.csv"

LEVEL1_GROUPS = {
    "Phocidae": ["Ross_Seal", "Harp_Seal", "Bearded_Seal", "Leopard_Seal"],
    "Monodontidae": ["Beluga,_White_Whale", "Narwhal"],
    "Balaenidae": ["Bowhead_Whale", "Northern_Right_Whale", "Southern_Right_Whale"],
    "Delphinidae": ["Common_Dolphin", "Bottlenose_Dolphin", "Long-Finned_Pilot_Whale"],
}
LEVEL1_GROUP_ORDER = ["Balaenidae", "Delphinidae", "Monodontidae", "Phocidae"]
LEVEL3_SPECIES = "Spinner_Dolphin"

print("Level 1 groups:")
for fam, sps in LEVEL1_GROUPS.items():
    print(f"  {fam}: {sps}")


## Load dataset & metadata


In [ ]:
print("Loading dataset and metadata...")
ds = load_from_disk(str(HF_LOCAL))
ds = ds.cast_column("audio", Audio(decode=False))
meta = pd.read_csv(METADATA_CSV)
print(f"  train: {len(ds['train'])} clips, test: {len(ds['test'])} clips")
print(f"  metadata rows: {len(meta)}")
meta.head()


## Build per-level filters


In [ ]:
def filter_for_level(meta, level):
    if level == 1:
        keep = []
        for fam, species_list in LEVEL1_GROUPS.items():
            mask = (meta["family"] == fam) & (meta["species"].isin(species_list))
            keep.append(meta[mask])
        return pd.concat(keep, ignore_index=False)
    if level == 2:
        return meta[meta["family"] == "Delphinidae"].copy()
    if level == 3:
        return meta[meta["species"] == LEVEL3_SPECIES].copy()
    raise ValueError(level)

keep_l1 = set(filter_for_level(meta, 1).index.tolist())
keep_l2 = set(filter_for_level(meta, 2).index.tolist())
keep_l3 = set(filter_for_level(meta, 3).index.tolist())

label_names_l1 = list(LEVEL1_GROUP_ORDER)
label_names_l2 = sorted(meta[meta["family"] == "Delphinidae"]["species"].unique().tolist())
l2_label_to_idx = {s: i for i, s in enumerate(label_names_l2)}

print(f"Level 1: {len(keep_l1)} clips")
print(f"Level 2: {len(keep_l2)} clips ({len(label_names_l2)} species)")
print(f"Level 3: {len(keep_l3)} clips")


## Preprocess every clip and accumulate per-level windows

For each clip we call `preprocess_clip()` (resample → 2-sec windows → bandpass)
and then extract the level-appropriate feature vectors and the (shared) Mel
spectrogram. Clips with sampling rates below 1 kHz are flagged because the
bandpass band would collapse against the Nyquist frequency.


In [ ]:
def iter_clips(ds, meta):
    lookup = {}
    for gid, row in meta.iterrows():
        lookup[(row["split"], int(row["index"]))] = (
            int(gid), str(row["species"]), str(row["family"])
        )
    for split_name in ("train", "test"):
        split = ds[split_name]
        for i in range(len(split)):
            sample = split[i]
            data, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
            if data.ndim > 1:
                data = data.mean(axis=1)
            data = np.asarray(data, dtype=np.float32)
            gid, species, family = lookup[(split_name, i)]
            yield {
                "global_clip_id": gid,
                "species": species,
                "family": family,
                "sampling_rate": int(sr),
                "audio": data,
                "is_train": split_name == "train",
            }

storage = {lvl: {"X": [], "S": [], "labels": [], "clip_ids": [],
                 "species": [], "is_train": []} for lvl in (1, 2, 3)}
low_sr = []

t0 = time.time()
for n_done, clip in enumerate(iter_clips(ds, meta), start=1):
    gid = clip["global_clip_id"]
    in_l1 = gid in keep_l1
    in_l2 = gid in keep_l2
    in_l3 = gid in keep_l3
    if not (in_l1 or in_l2 or in_l3):
        continue
    sr = clip["sampling_rate"]
    if sr < 1000:
        low_sr.append((gid, clip["species"], sr))
    windows = preprocess_clip(clip["audio"], orig_sr=sr)
    if not windows:
        continue
    for w in windows:
        mel = compute_mel_spectrogram(w, sr=TARGET_SR)
        if in_l1:
            storage[1]["X"].append(extract_features_level1(w))
            storage[1]["S"].append(mel)
            storage[1]["labels"].append(LEVEL1_GROUP_ORDER.index(clip["family"]))
            storage[1]["clip_ids"].append(gid)
            storage[1]["species"].append(clip["species"])
            storage[1]["is_train"].append(clip["is_train"])
        if in_l2:
            storage[2]["X"].append(extract_features_level2(w))
            storage[2]["S"].append(mel)
            storage[2]["labels"].append(l2_label_to_idx[clip["species"]])
            storage[2]["clip_ids"].append(gid)
            storage[2]["species"].append(clip["species"])
            storage[2]["is_train"].append(clip["is_train"])
        if in_l3:
            storage[3]["X"].append(extract_features_level3(w))
            storage[3]["S"].append(mel)
            storage[3]["labels"].append(-1)
            storage[3]["clip_ids"].append(gid)
            storage[3]["species"].append(clip["species"])
            storage[3]["is_train"].append(clip["is_train"])
    if n_done % 200 == 0:
        print(f"  processed {n_done} clips, {time.time() - t0:.1f}s")

print(f"\nTotal time: {time.time() - t0:.1f}s")
print(f"Low-sampling-rate clips (sr < 1000 Hz): {len(low_sr)}")
for gid, sp, sr in low_sr[:10]:
    print(f"  clip_id={gid} species={sp} sr={sr} Hz")


## Save .npz files per level


In [ ]:
label_names_by_level = {
    1: np.array(label_names_l1, dtype=object),
    2: np.array(label_names_l2, dtype=object),
    3: np.array([], dtype=object),
}
feat_paths = {
    1: PROJECT_ROOT / "Level1" / "Data" / "level1_features.npz",
    2: PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz",
    3: PROJECT_ROOT / "Level3" / "Data" / "level3_features.npz",
}
spec_paths = {
    1: PROJECT_ROOT / "Level1" / "Data" / "level1_spectrograms.npz",
    2: PROJECT_ROOT / "Level2" / "Data" / "level2_spectrograms.npz",
    3: PROJECT_ROOT / "Level3" / "Data" / "level3_spectrograms.npz",
}

summary = {}
for level in (1, 2, 3):
    s = storage[level]
    X = np.stack(s["X"]).astype(np.float32)
    S_arr = np.stack(s["S"]).astype(np.float32)
    labels = np.array(s["labels"], dtype=np.int64)
    clip_ids = np.array(s["clip_ids"], dtype=np.int64)
    species_arr = np.array(s["species"], dtype=object)
    is_train = np.array(s["is_train"], dtype=bool)
    label_names = label_names_by_level[level]

    np.savez_compressed(
        feat_paths[level],
        X=X, labels=labels, clip_ids=clip_ids,
        label_names=label_names, species=species_arr, is_train=is_train,
    )
    np.savez_compressed(
        spec_paths[level],
        S=S_arr, labels=labels, clip_ids=clip_ids,
        label_names=label_names, is_train=is_train,
    )
    summary[level] = {
        "n_windows": int(X.shape[0]),
        "feature_dim": int(X.shape[1]),
        "spec_shape": tuple(S_arr.shape),
        "n_clips": int(np.unique(clip_ids).size),
    }
    print(f"Level {level}: {X.shape[0]} windows, dim={X.shape[1]}, "
          f"spec={S_arr.shape}, n_clips={np.unique(clip_ids).size}")


## Summary statistics


In [ ]:
for level in (1, 2, 3):
    s = storage[level]
    labels = np.array(s["labels"], dtype=np.int64)
    print(f"\n=== Level {level} ===")
    print(f"  windows: {summary[level]['n_windows']}")
    print(f"  feature dim: {summary[level]['feature_dim']}")
    print(f"  spectrogram shape: {summary[level]['spec_shape']}")
    print(f"  unique clips: {summary[level]['n_clips']}")
    if level == 3:
        print(f"  unlabeled (Spinner Dolphin only)")
    else:
        names = label_names_by_level[level]
        unique, counts = np.unique(labels, return_counts=True)
        for u, c in zip(unique, counts):
            print(f"  {names[u]}: {c}")
print("\nAll .npz files written.")
